<a href="https://colab.research.google.com/github/quarcs-lab/gdo4bolivia/blob/main/notebooks/esda_heterogeneity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spatial heterogeneity: a multiscale GWR (MGWR) of poverty on night-time lights

Part of the [GeoDevelopment Observatory of Bolivia](https://quarcs-lab.github.io/gdo4bolivia/).

Night-time lights are a common proxy for economic activity. Is their link with poverty the same everywhere in Bolivia? This notebook relates **multidimensional poverty** (`sdg1_2_mpi`, 0–1; **higher = poorer**) from the Atlas de los ODS en Bolivia 2026 to **log night-time lights per capita** (`ln_t400NTLpc`) and lets the relationship vary over space with a multiscale geographically weighted regression (MGWR). It does so twice: poverty around the 2012 census on lights in 2012, and poverty around the 2024 census on lights in 2020, the last year of the lights series.

Every step is one function of the [`geo4bolivia`](https://quarcs-lab.github.io/geo4bolivia/) package:

1. **Load** the two datasets and pair their years
2. **A global benchmark**: one slope for the whole country ([`geo.regression`](https://quarcs-lab.github.io/geo4bolivia/reference/regression.html)), and a test of what it leaves in its residuals ([`geo.spatial_diagnostics`](https://quarcs-lab.github.io/geo4bolivia/reference/spatial_diagnostics.html))
3. **MGWR in 2012** ([`geo.mgwr`](https://quarcs-lab.github.io/geo4bolivia/reference/mgwr.html)): bandwidths and their intervals, coefficient maps, significance, local collinearity, and fit against global models
4. **MGWR in 2024**, and what changed
5. **Summary**

## 0. Setup

In Google Colab the next cell installs the package with its MGWR extra (it brings the mgwr library; Colab already has the rest). Locally, `uv sync` has installed everything already.

In [1]:
import importlib.util

if any(importlib.util.find_spec(p) is None for p in ["geo4bolivia", "mgwr"]):
    !pip install -q "geo4bolivia[mgwr]"

In [2]:
import os
import sys
import warnings

import pandas as pd
import plotly.express as px
import plotly.io as pio

import geo4bolivia as geo

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"     # also in the worker processes that MGWR starts
# Plotly: Colab's own renderer there; elsewhere load plotly.js from its CDN to keep the file small
pio.renderers.default = "colab" if "google.colab" in sys.modules else "notebook_connected"
print("geo4bolivia", geo.__version__)

geo4bolivia 0.7.2


## 1. Load the data

Poverty comes from the SDG Atlas, which gives each of the 343 municipalities and indigenous autonomies two values: one near the 2012 census and one near the 2024 census, stored as the years 2012 and 2024. Night-time lights per capita (`ntl`, 2012–2020) are built from gridded population on the older municipal map of 2015, `map339`, so they cover its **339 units**: the four units created since (three indigenous autonomies, TIOC, and San Pedro de Macha) have no lights value, and the six municipalities that ceded territory to them are described by their 2015 territory. The two maps share geo IDs, so the datasets join on `geo_id`; the spatial steps use the 2015 map (`map="map339"`), the polygons the lights belong to.

`ln_t400NTLpc` is the Hodrick–Prescott trend of the log of lights per capita, which removes year-to-year noise. The lights of 2012 go with the poverty of 2012, and the lights of 2020 with the poverty of 2024: the lights of 2020 are stored below in the rows of 2024, so that each period is one "year" of the panel.

In [3]:
mpi = geo.load("sdg01", variables="sdg1_2_mpi", groups=["department"])
ntl = geo.load("ntl", variables="ln_t400NTLpc", years=[2012, 2020])
ntl["year"] = ntl["year"].replace({2020: 2024})     # lights of 2020 next to poverty of ~2024

df = mpi.merge(ntl, on=["geo_id", "year"])          # inner join: the 339 units with lights
# lights belong to the polygons of the 2015 map, so the spatial steps below use map="map339"
no_lights = mpi.loc[~mpi["geo_id"].isin(ntl["geo_id"]), "name"].unique()
print(df["geo_id"].nunique(), "units | without lights:", ", ".join(no_lights))
df.head()

339 units | without lights: TIOC-Raqaypampa, TIOC-Territorio Indígena Multiétnico, TIOC-Jatun Ayllu Yura, San Pedro de Macha


,geo_id,name,year,department,sdg1_2_mpi,ln_t400NTLpc
0,BO-01-01-01,Azurduy,2012,Chuquisaca,0.512,11.048312
1,BO-01-01-01,Azurduy,2024,Chuquisaca,0.187,13.136794
2,BO-01-01-02,Tarvita,2012,Chuquisaca,0.452,12.329279
3,BO-01-01-02,Tarvita,2024,Chuquisaca,0.133,12.656998
4,BO-01-02-01,Villa Serrano,2012,Chuquisaca,0.267,12.815315


In [4]:
geo.describe(df)

,variable,label,unit,year,count,mean,std,min,p25,median,p75,max
0,sdg1_2_mpi,Multidimensional poverty index (0-1),index (0-1),2012,339,0.231917,0.103422,0.018000,0.152500,0.226000,0.301500,0.541000
1,sdg1_2_mpi,Multidimensional poverty index (0-1),index (0-1),2024,339,0.062015,0.041915,0.003000,0.030000,0.053000,0.081500,0.223000
2,ln_t400NTLpc,Log night-time lights per capita (trend),log(10⁸ × nW/cm²/sr per person),2012,339,13.033028,1.671621,0.856095,12.050438,13.118243,14.268402,17.071051
3,ln_t400NTLpc,Log night-time lights per capita (trend),log(10⁸ × nW/cm²/sr per person),2024,339,14.039931,1.177322,10.488098,13.205945,14.101274,14.922364,18.063404


Between the two periods poverty fell sharply and its spread narrowed, while lights per capita rose and their spread (in logs) narrowed too. The darkest unit in 2012 is far from the rest: a log of 0.86, against a first quartile of 12.1.

## 2. A global benchmark

### One slope for the whole country

`geo.scatter` plots the two variables with the least-squares line, and `geo.regression` gives the same line as a regression table with robust standard errors.

In [5]:
geo.scatter(df, "ln_t400NTLpc", "sdg1_2_mpi", year=2012, color="department")

Across 339 units in 2012, **Multidimensional poverty index (0-1)** and **Log night-time lights per capita (trend)** are negatively correlated (r = -0.54, a moderate association): units with higher Log night-time lights per capita (trend) tend to have lower Multidimensional poverty index (0-1).

The least-squares line has a slope of -0.0335: one more unit of Log night-time lights per capita (trend) goes with 0.0335 lower Multidimensional poverty index (0-1), on average. The line accounts for 29% of the variation in Multidimensional poverty index (0-1) across units (R² = 0.29); the slope is statistically significant at the 1% level, by a test that treats the units as independent of each other.

_These are associations, not causal effects. A causal reading needs a research design._

In [6]:
ols = geo.regression(df, "sdg1_2_mpi", "ln_t400NTLpc", year=2012)
ols.gt

GT(_tbl_data=                                       term        (1)
0  Log night-time lights per capita (trend)  -0.034***
1                                              (0.005)
2                                 Intercept   0.669***
3                                              (0.063)
4                              Observations        339
5                                        R²      0.294
6                               Adjusted R²      0.292, _body=<great_tables._gt_data.Body object at 0x12d2c95d0>, _boxhead=Boxhead([ColInfo(var='term', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None, stub_level=None), ColInfo(var='(1)', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x12d2bb290>, _spanners=Spanners([SpannerInfo(spanner_id='Multidimensional poverty index (0-1)', spanner_level=0, spanner_label='Multidimensional poverty index (0-1)', spanner_units=None, spanner_pattern=None, vars=['(1)'], built=None)]), _heading=Heading(title='Regression of Multidimensional poverty index (0-1)', subtitle='Ordinary least squares, 2012', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12d2c90d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12d2c9250>, _source_notes=['*** p<0.01, ** p<0.05, * p<0.10. Standard errors in parentheses: heteroskedasticity-robust (HC1).'], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=[4], mask=None), grpname=None, colname='term', rownum=4, colnum=None, styles=[CellStyleBorders(sides='top', color='#7f7f7f', style='solid', weight='1px')]), StyleInfo(locname=LocBody(columns=None, rows=[4], mask=None), grpname=None, colname='(1)', rownum=4, colnum=None, styles=[CellStyleBorders(sides='top', color='#7f7f7f', style='solid', weight='1px')])], _locale=<great_tables._gt_data.Locale object at 0x12d2c93d0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'

Across the country, brighter municipalities are less poor. The slope is −0.034: one more log point of lights per capita (about 2.7 times more light per person) goes with an MPI 0.034 lower, about a third of the standard deviation of poverty across these units (0.103). Lights alone account for 29% of the variation in poverty (R² = 0.294). The cloud of points is wide, which suggests that one line may not fit every region.

### What the global model leaves out

If one line described every part of the country, its residuals would be scattered over the map at random. `geo.spatial_diagnostics` tests that: Moran's I of the residuals, and the Lagrange multiplier tests for the two standard spatial models.

In [7]:
diag = geo.spatial_diagnostics(df, "sdg1_2_mpi", "ln_t400NTLpc", year=2012, map="map339")
diag

GT(_tbl_data=                    Test Statistic  df p-value
0  Moran's I (residuals)    0.3517  --  0.0000
1                 LM lag  110.8661   1  0.0000
2               LM error  103.7109   1  0.0000
3          Robust LM lag   11.2724   1  0.0008
4        Robust LM error    4.1172   1  0.0424
5               LM SARMA  114.9833   2  0.0000, _body=<great_tables._gt_data.Body object at 0x13024b5d0>, _boxhead=Boxhead([ColInfo(var='Test', type=<ColInfoTypeEnum.stub: 2>, column_label='Test', column_align='left', column_width=None, stub_level=0), ColInfo(var='Statistic', type=<ColInfoTypeEnum.default: 1>, column_label='Statistic', column_align='right', column_width=None, stub_level=None), ColInfo(var='df', type=<ColInfoTypeEnum.default: 1>, column_label='df', column_align='right', column_width=None, stub_level=None), ColInfo(var='p-value', type=<ColInfoTypeEnum.default: 1>, column_label='p-value', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x12e8d1810>, _spanners=Spanners([]), _heading=Heading(title='Spatial dependence diagnostics for Multidimensional poverty index (0-1)', subtitle='OLS residual tests | W: queen contiguity (1 island(s) attached to nearest neighbor), row-standardized, n=339 (on map339)', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12f7ec550>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12ebf0bd0>, _source_notes=['Anselin-Florax recommendation: lag'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x13024b390>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='

Moran's I on the OLS residuals is **0.352** (statistically significant at the 1% level) under queen contiguity (1 island(s) attached to nearest neighbor), row-standardized, n=339 (on map339) — the residuals cluster in space, so the non-spatial model leaves spatial structure on the table.

At α = 0.05, the tests that reject no-spatial-dependence are: LM lag, LM error, robust LM lag, robust LM error. The *robust* forms are the decisive ones — each tests one dependence channel while allowing for the other.

**Recommendation: `lag`** — Both robust LM tests reject at alpha = 0.05 (robust LM lag = 11.27, robust LM error = 4.12), so the larger robust statistic wins and points to the lag model. With both channels active, consider durbin (SDM) - it nests both the lag and the spatially-lagged-X channels and lets the data separate them.

_These are associations, not causal effects. A causal reading needs a research design._

The residuals are clearly clustered: their Moran's I is 0.352, and every test rejects the absence of spatial dependence. Neighbouring municipalities are poorer, or less poor, than their lights predict *together*. The tests point to a spatial lag model, which would keep one slope and add the neighbours' poverty. This notebook asks a different question: whether the slope itself changes from place to place.

## 3. MGWR in 2012

A geographically weighted regression fits a regression at every municipality, giving more weight to the municipalities near it. Each coefficient becomes a surface that can be mapped. The **bandwidth** is the number of nearest municipalities that enter each local regression: a small bandwidth means a relationship that changes over short distances, and a bandwidth close to the number of units (339) one that is about the same everywhere.

The multiscale version (MGWR) finds **one bandwidth per term**, here one for the intercept and one for the slope on lights. `geo.mgwr` follows the workflow of the [mgwr library](https://github.com/pysal/mgwr): the variables are z-standardized, so the coefficients are in standard deviations; the kernel is an adaptive bisquare; the bandwidths minimise the AICc; and the local t-tests of each term are corrected for multiple testing. One fit takes about half a minute.

In [8]:
mg = geo.mgwr(df, "sdg1_2_mpi", "ln_t400NTLpc", year=2012, map="map339")
print(mg.interpret())

MGWR let every term in the local regression of **Multidimensional poverty index (0-1)** operate at its own spatial scale (variables z-standardized, 339 units); overall R² is 0.647.

Selected bandwidths: **the intercept** at 44 nearest neighbors (local); **Log night-time lights per capita (trend)** at 65 nearest neighbors (local). Smaller bandwidths mean the association shifts over shorter distances.

The **Log night-time lights per capita (trend)** association varies locally from -1.35 to -0.165; it is strongest around **Colomi** (local coefficient -1.35). 260 of 339 units clear the corrected threshold (|t| ≥ 2.92); non-significant units are greyed on the map.

Coefficients are on the standardized scale (MGWR requires z-standardized variables), so magnitudes are comparable across terms; significance uses per-term corrected alphas.

_These are associations, not causal effects. A causal reading needs a research design._


### Bandwidths and their intervals

In [9]:
mg.gt_bw

GT(_tbl_data=                                       Term Bandwidth 95% interval    ENP  \
0                                 Intercept        44     44 to 48  20.93   
1  Log night-time lights per capita (trend)        65     64 to 85  13.47   

  Corrected alpha Critical |t|  
0        0.002389         3.06  
1        0.003712        2.922  , _body=<great_tables._gt_data.Body object at 0x1316f6290>, _boxhead=Boxhead([ColInfo(var='Term', type=<ColInfoTypeEnum.stub: 2>, column_label='Term', column_align='left', column_width=None, stub_level=0), ColInfo(var='Bandwidth', type=<ColInfoTypeEnum.default: 1>, column_label='Bandwidth', column_align='right', column_width=None, stub_level=None), ColInfo(var='95% interval', type=<ColInfoTypeEnum.default: 1>, column_label='95% interval', column_align='left', column_width=None, stub_level=None), ColInfo(var='ENP', type=<ColInfoTypeEnum.default: 1>, column_label='ENP', column_align='right', column_width=None, stub_level=None), ColInfo(var='Corrected alpha', type=<ColInfoTypeEnum.default: 1>, column_label='Corrected alpha', column_align='right', column_width=None, stub_level=None), ColInfo(var='Critical |t|', type=<ColInfoTypeEnum.default: 1>, column_label='Critical |t|', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x1316f5a50>, _spanners=Spanners([]), _heading=Heading(title='MGWR bandwidths', subtitle='adaptive nearest-neighbor bandwidths out of 339 units', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1316f5f10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1316f6b10>, _source_notes=['Smaller bandwidths mean the association varies over shorter distances (a more local process); a bandwidth near n is effectively global.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1316f6b50>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', 

In [10]:
print("Bandwidths:", mg.bw)
print("95% intervals:", mg.bw_interval)

Bandwidths: {'const': 44.0, 'ln_t400NTLpc': 65.0}
95% intervals: {'const': (44.0, 48.0), 'ln_t400NTLpc': (64.0, 85.0)}


MGWR selects **44 nearest municipalities for the intercept** (95% interval 44 to 48) and **65 for the slope on lights** (64 to 85). Both are far below 339, so both the level of poverty at given lights and its link with lights are local processes. The slope varies over a somewhat larger scale than the intercept, and the two intervals do not overlap: a single bandwidth for both terms would have hidden that difference.

One limit to know: the search of the mgwr library does not try very small bandwidths. With two terms the smallest it considers is 44, so read the bandwidth of the intercept as "as local as the search goes": the true scale may be smaller.

### The coefficient maps

`mg.figs` holds one map per term. Municipalities whose coefficient is not significant after the correction are grey.

In [11]:
mg.figs["ln_t400NTLpc"]

In [12]:
mg.figs["const"]

In [13]:
geo.distribution(mg.df, "ln_t400NTLpc_coef", by="department", kind="box",
                 title="Local slope on lights by department, 2012").fig

In [14]:
local = geo.add_names(mg.df, groups=["department"])
by_dep = local.groupby("department", as_index=False).agg(
    units=("geo_id", "size"),
    mean_slope=("ln_t400NTLpc_coef", "mean"), sd_slope=("ln_t400NTLpc_coef", "std"),
    significant_slopes=("ln_t400NTLpc_significant", "sum"),
    mean_intercept=("const_coef", "mean"))
by_dep.sort_values("mean_slope").round(2)

,department,units,mean_slope,sd_slope,significant_slopes,mean_intercept
2,Cochabamba,47,-1.23,0.17,47,0.38
1,Chuquisaca,29,-1.02,0.23,29,0.42
6,Potosí,40,-0.92,0.44,35,0.18
4,Oruro,35,-0.70,0.28,26,0.25
8,Tarija,11,-0.57,0.12,11,-0.12
7,Santa Cruz,56,-0.47,0.13,21,-0.30
3,La Paz,87,-0.41,0.08,74,-0.26
0,Beni,19,-0.29,0.14,6,0.03
5,Pando,15,-0.28,0.04,11,0.46


In [15]:
def names(table, column, n=5):
    return ", ".join(f"{a} ({b}, {c:.2f})" for a, b, c in
                     zip(table["name"].head(n), table["department"], table[column]))

print("Steepest slopes:", names(local.nsmallest(5, "ln_t400NTLpc_coef"), "ln_t400NTLpc_coef"))
print("Flattest slopes:", names(local.nlargest(5, "ln_t400NTLpc_coef"), "ln_t400NTLpc_coef"))
print("Highest intercepts:", names(local.nlargest(5, "const_coef"), "const_coef"))
print("Lowest intercepts:", names(local.nsmallest(5, "const_coef"), "const_coef"))

Steepest slopes: Colomi (Cochabamba, -1.35), Sacaba (Cochabamba, -1.34), Cochabamba (Cochabamba, -1.34), Puna (Potosí, -1.34), Colcapirhua (Cochabamba, -1.34)
Flattest slopes: Baures (Beni, -0.17), Magdalena (Beni, -0.18), Huacaraje (Beni, -0.19), San Pedro de Quemes (Potosí, -0.20), San Javier (Beni, -0.21)
Highest intercepts: El Villar (Chuquisaca, 0.64), Monteagudo (Chuquisaca, 0.63), Villa Vaca Guzmán (Chuquisaca, 0.61), Tarvita (Chuquisaca, 0.60), Camiri (Santa Cruz, 0.60)
Lowest intercepts: Nuestra Señora de La Paz (La Paz, -0.81), Coroico (La Paz, -0.79), Guanay (La Paz, -0.75), Coripata (La Paz, -0.72), Caranavi (La Paz, -0.71)


- **The sign never changes.** Every one of the 339 local slopes is negative, from −1.35 to −0.17 standard deviations of poverty per standard deviation of lights. The global slope on the same scale is the correlation of section 2, −0.54. More light goes with less poverty everywhere; what varies is the strength, about eightfold.
- **The slope is steepest in the Andean valleys.** The department averages are −1.23 in Cochabamba, −1.02 in Chuquisaca and −0.92 in Potosí, and the steepest slopes of all are around the city of Cochabamba (Colomi, Sacaba, Cochabamba, Colcapirhua: about −1.34) and in Puna, Potosí.
- **It is flattest in the Amazonian north.** Beni averages −0.29 and Pando −0.28, with Baures, Magdalena and Huacaraje in Beni the flattest (−0.17 to −0.19). The slopes of La Paz (−0.41) and Santa Cruz (−0.47) are also weaker than the global one.
- **Potosí and Oruro are split.** Their slopes spread more than those of any other department (standard deviations 0.44 and 0.28). On the map they are steep next to the valleys and flat in the south-west of Potosí and the west of Oruro.
- **The intercept map** shows the level of poverty at average lights, in standard deviations. On average it is positive (poorer than the lights suggest) in Pando (0.46), Chuquisaca (0.42) and Cochabamba (0.38), and negative in La Paz (−0.26) and Santa Cruz (−0.30). The highest intercepts are in the south-east of Chuquisaca and the neighbouring Chaco of Santa Cruz, and the lowest in the city of La Paz and the Yungas north of it.

### Significance and local collinearity

Each term has its own critical value, because a term with a smaller bandwidth runs more independent local tests. `local_cn` is the condition number of each local regression: above about 30, the local coefficients cannot be told apart from each other.

In [16]:
print("Critical |t| after correction:", {k: round(v, 2) for k, v in mg.critical_t.items()})
print("Significant slopes:", mg.df["ln_t400NTLpc_significant"].sum(), "of", len(mg.df),
      "| significant intercepts:", mg.df["const_significant"].sum())
print("Slopes with |t| > 1.96:", (mg.df["ln_t400NTLpc_t"].abs() > 1.96).sum())
print("Negative slopes:", (mg.df["ln_t400NTLpc_coef"] < 0).sum())
print("Local condition number: median", round(mg.df["local_cn"].median(), 2),
      "| maximum", round(mg.df["local_cn"].max(), 2))

Critical |t| after correction: {'const': 3.06, 'ln_t400NTLpc': 2.92}
Significant slopes: 260 of 339 | significant intercepts: 134
Slopes with |t| > 1.96: 330
Negative slopes: 339
Local condition number: median 1.43 | maximum 4.93


After the correction (critical |t| of 2.92 for the slope, against the usual 1.96), **260 of the 339 slopes are significant**; without it, 330 would be. The table above gives them by department: all the slopes of Chuquisaca, Cochabamba and Tarija are significant, and most of those of Potosí (35 of 40) and La Paz (74 of 87). Fewer than half are in Santa Cruz (21 of 56) and Beni (6 of 19), where the slopes are also flat.

The local condition numbers are small (median 1.4, maximum 4.9, far below 30): with one regressor, the intercept and the slope are well identified everywhere.

### Fit against global models

Is the local model worth its complexity? R² does not depend on the units of the variables, but the AICc does, and the AICc of MGWR refers to the z-standardized variables. So the comparison fits the global regression on the same standardized variables, and computes its AICc from the fitted statsmodels object (`.models`) with the formula the mgwr library uses. The package has no function for that number.

The tests of section 2 pointed to a spatial lag model, which keeps one slope and adds the neighbours' poverty. `geo.spatial_model` fits it on the same variables, as a second global benchmark.

In [17]:
def standardized(data, columns):
    z = data.copy()
    z[columns] = (z[columns] - z[columns].mean()) / z[columns].std(ddof=0)
    return z

def aicc(fit):                        # AICc of a least-squares fit, as in the mgwr library
    n, k = int(fit.nobs), len(fit.params)
    return -2 * fit.llf + 2 * n * (k + 1) / (n - k - 2)

def compare(year, mg):
    """Global models on the z-standardized variables of one period, against MGWR."""
    z = standardized(df[df["year"] == year], ["sdg1_2_mpi", "ln_t400NTLpc"])
    ols_z = geo.regression(z, "sdg1_2_mpi", "ln_t400NTLpc")
    lag = geo.spatial_model(z, "sdg1_2_mpi", "ln_t400NTLpc", model="lag", map="map339")
    print(f"Global standardized slope: {ols_z.df['estimate'][0]:.2f} | "
          f"spatial lag model: AIC {lag.aic:.1f}, rho {lag.rho:.2f}")
    table = pd.DataFrame(
        {"least squares": [ols_z.summary["r2"][0], ols_z.summary["r2_adj"][0],
                           aicc(ols_z.models[0]), len(ols_z.models[0].params)],
         "MGWR": [mg.r2, mg.model_obj.adj_R2, mg.aicc, mg.model_obj.ENP]},
        index=["R²", "adjusted R²", "AICc", "effective parameters"])
    return table.round(3)

compare(2012, mg)

Global standardized slope: -0.54 | spatial lag model: AIC 751.6, rho 0.59


,least squares,MGWR
R²,0.294,0.647
adjusted R²,0.292,0.607
AICc,850.255,688.645
effective parameters,2.000,34.396


In [18]:
resid = geo.moran(mg.df, "residual", map="map339")
print(f"Moran's I of the residuals: least squares {diag.moran_i_resid:.3f}, "
      f"MGWR {resid.moran_i:.3f} (pseudo p = {resid.p_sim:.3f})")

Moran's I of the residuals: least squares 0.352, MGWR 0.089 (pseudo p = 0.004)


- **Against least squares.** Letting the intercept and the slope vary raises R² from 0.294 to 0.647 (adjusted: 0.292 to 0.607) and lowers the AICc from 850.3 to 688.6. The AICc penalises the extra flexibility: MGWR uses the equivalent of 34.4 parameters, against 2.
- **Against the spatial lag model.** A single slope with a spatial lag has an AIC of 751.6: better than least squares, and well above the 688.6 of MGWR. The two criteria count parameters slightly differently, so the comparison is approximate, but the gap is wide. In 2012, coefficients that vary over space describe these data better than one slope with spatial dependence.
- **Residuals.** MGWR absorbs most of the spatial pattern that least squares left in its residuals: their Moran's I falls from 0.352 to 0.089, small but still distinguishable from zero (pseudo p = 0.004). `mg.fig` maps them.

## 4. MGWR in 2024

The Atlas measures poverty again around the 2024 census; the lights series ends in 2020. The same steps, on the same 339 units, show whether the link has changed. Because MGWR standardizes the variables within each period, the slopes of the two periods are comparable even though poverty fell a lot.

In [19]:
ols24 = geo.regression(df, "sdg1_2_mpi", "ln_t400NTLpc", year=2024)
diag24 = geo.spatial_diagnostics(df, "sdg1_2_mpi", "ln_t400NTLpc", year=2024, map="map339")
slope24 = ols24.df.set_index("term").loc["ln_t400NTLpc"]
corr24 = geo.scatter(df, "ln_t400NTLpc", "sdg1_2_mpi", year=2024).corr
print(f"Global slope {slope24['estimate']:.4f} (SE {slope24['std_error']:.4f}), "
      f"R² {ols24.summary.loc[0, 'r2']:.3f}, correlation {corr24:.2f}")
print(f"Moran's I of its residuals {diag24.moran_i_resid:.3f}; the tests point to: {diag24.recommendation}")

Global slope -0.0180 (SE 0.0018), R² 0.255, correlation -0.50
Moran's I of its residuals 0.337; the tests point to: lag


In [20]:
mg24 = geo.mgwr(df, "sdg1_2_mpi", "ln_t400NTLpc", year=2024, map="map339")
mg24.gt_bw

GT(_tbl_data=                                       Term Bandwidth 95% interval    ENP  \
0                                 Intercept        44     44 to 48  21.66   
1  Log night-time lights per capita (trend)       101   101 to 112  8.214   

  Corrected alpha Critical |t|  
0        0.002309        3.071  
1        0.006087         2.76  , _body=<great_tables._gt_data.Body object at 0x131b62710>, _boxhead=Boxhead([ColInfo(var='Term', type=<ColInfoTypeEnum.stub: 2>, column_label='Term', column_align='left', column_width=None, stub_level=0), ColInfo(var='Bandwidth', type=<ColInfoTypeEnum.default: 1>, column_label='Bandwidth', column_align='right', column_width=None, stub_level=None), ColInfo(var='95% interval', type=<ColInfoTypeEnum.default: 1>, column_label='95% interval', column_align='left', column_width=None, stub_level=None), ColInfo(var='ENP', type=<ColInfoTypeEnum.default: 1>, column_label='ENP', column_align='right', column_width=None, stub_level=None), ColInfo(var='Corrected alpha', type=<ColInfoTypeEnum.default: 1>, column_label='Corrected alpha', column_align='right', column_width=None, stub_level=None), ColInfo(var='Critical |t|', type=<ColInfoTypeEnum.default: 1>, column_label='Critical |t|', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x131b61cd0>, _spanners=Spanners([]), _heading=Heading(title='MGWR bandwidths', subtitle='adaptive nearest-neighbor bandwidths out of 339 units', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x131b63150>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x131b63190>, _source_notes=['Smaller bandwidths mean the association varies over shorter distances (a more local process); a bandwidth near n is effectively global.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x131b631d0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', 

In [21]:
mg24.figs["ln_t400NTLpc"]

In [22]:
resid24 = geo.moran(mg24.df, "residual", map="map339")
print("Bandwidths:", mg24.bw, "| 95% intervals:", mg24.bw_interval)
print("Significant slopes:", mg24.df["ln_t400NTLpc_significant"].sum(), "of", len(mg24.df),
      "| negative slopes:", (mg24.df["ln_t400NTLpc_coef"] < 0).sum(),
      "| range:", mg24.df["ln_t400NTLpc_coef"].min().round(2), "to",
      mg24.df["ln_t400NTLpc_coef"].max().round(2))
print("Local condition number: maximum", round(mg24.df["local_cn"].max(), 2))
print(f"Moran's I of the residuals: least squares {diag24.moran_i_resid:.3f}, "
      f"MGWR {resid24.moran_i:.3f} (pseudo p = {resid24.p_sim:.3f})")
compare(2024, mg24)

Bandwidths: {'const': 44.0, 'ln_t400NTLpc': 101.0} | 95% intervals: {'const': (44.0, 48.0), 'ln_t400NTLpc': (101.0, 112.0)}
Significant slopes: 248 of 339 | negative slopes: 339 | range: -0.96 to -0.17
Local condition number: maximum 3.2
Moran's I of the residuals: least squares 0.337, MGWR 0.072 (pseudo p = 0.014)


Global standardized slope: -0.50 | spatial lag model: AIC 784.2, rho 0.56


,least squares,MGWR
R²,0.255,0.541
adjusted R²,0.253,0.496
AICc,868.354,766.378
effective parameters,2.000,29.870


### The local slopes in both periods

In [23]:
both = mg.df.merge(mg24.df, on=["geo_id", "name"], suffixes=("_2012", "_2024"))
both = geo.add_names(both, groups=["department"])
r = both["ln_t400NTLpc_coef_2012"].corr(both["ln_t400NTLpc_coef_2024"])
print(f"Correlation of the local slopes of the two periods: {r:.2f}")

px.scatter(both, x="ln_t400NTLpc_coef_2012", y="ln_t400NTLpc_coef_2024", color="department",
           hover_name="name", title="Local slopes on lights in both periods",
           labels={"ln_t400NTLpc_coef_2012": "Local slope, poverty 2012 on lights 2012",
                   "ln_t400NTLpc_coef_2024": "Local slope, poverty 2024 on lights 2020"})

Correlation of the local slopes of the two periods: 0.84


In [24]:
means = both.groupby("department", as_index=False)[
    ["ln_t400NTLpc_coef_2012", "ln_t400NTLpc_coef_2024"]].mean()
means = means.rename(columns={"ln_t400NTLpc_coef_2012": "2012", "ln_t400NTLpc_coef_2024": "2024"})
long = means.melt(id_vars="department", var_name="period", value_name="mean local slope")

px.bar(long, x="department", y="mean local slope", color="period", barmode="group",
       title="Mean local slope on lights by department").show()
means.sort_values("2012").round(2)

,department,2012,2024
2,Cochabamba,-1.23,-0.77
1,Chuquisaca,-1.02,-0.78
6,Potosí,-0.92,-0.84
4,Oruro,-0.70,-0.57
8,Tarija,-0.57,-0.74
7,Santa Cruz,-0.47,-0.34
3,La Paz,-0.41,-0.23
0,Beni,-0.29,-0.34
5,Pando,-0.28,-0.29


- **The global link persists and is a little weaker.** In 2024 the slope is −0.018 in MPI units, about half the slope of 2012, but poverty itself is far lower. On the standardized scale the association falls only from −0.54 to −0.50, and R² from 0.294 to 0.255. The residuals of the global model are as clustered as before (Moran's I 0.337).
- **It is still local.** MGWR keeps the intercept bandwidth at 44 (interval 44 to 48) and raises R² to 0.541 (adjusted 0.496), with an AICc of 766.4 against 868.4 for least squares. The Moran's I of the residuals falls to 0.072. The spatial lag model comes closer than in 2012: its AIC is 784.2.
- **The slope varies over a broader scale.** Its bandwidth rises from 65 to 101 nearest municipalities (interval 101 to 112, which does not overlap the 64 to 85 of 2012), and the range of the local slopes narrows: −0.96 to −0.17, against −1.35 to −0.17 in 2012. All 339 are again negative, 248 are significant after the correction, and no local condition number exceeds 3.2.
- **The map is much the same.** The two sets of local slopes correlate at 0.84. The steep slopes of the valleys moved towards the rest: Cochabamba's average goes from −1.23 to −0.77 and Chuquisaca's from −1.02 to −0.78, while Tarija's strengthens (−0.57 to −0.74). Potosí (−0.84) now has the steepest average slope, and La Paz (−0.23) the flattest.

Lights in 2020 precede poverty in 2024 by four years, so this is a comparison of patterns, not a test of how lights predict later poverty.

## Summary

- **One sign, many strengths.** In both periods, every one of the 339 local slopes is negative: brighter municipalities are less poor everywhere. The strength of the link varies about eightfold in 2012 (−1.35 to −0.17) and less in 2024 (−0.96 to −0.17).
- **The global model misses a spatial pattern.** One slope for the country accounts for 29% of the variation in poverty in 2012 and leaves clustered residuals (Moran's I = 0.352). MGWR accounts for 65%, has a much lower AICc than least squares and than a spatial lag model, and leaves residuals that are only weakly clustered (0.089).
- **Two local processes, at different scales.** The intercept has a bandwidth of 44 nearest municipalities in both periods, the smallest the search considers. The slope on lights has one of 65 in 2012 and 101 in 2024: the link became more alike across the country.
- **Where lights are informative.** As a proxy for poverty, night-time lights carry most information in the Andean valleys of Cochabamba, Chuquisaca and Potosí, and least in the Amazonian north (Beni, Pando), where the local slopes are flattest, and in Santa Cruz, where most are not significant. The companion notebook `embeddings_mpi_ml` tests whether satellite embeddings do better.
- **Caveats.** These are associations from a model with one explanatory variable, not causal effects. MGWR shows where an association differs, not why: local coefficients absorb omitted factors that vary over space (altitude, remoteness, access to markets). Each local regression rests on few units, and the 2024 fit pairs poverty with lights of four years earlier.

Related notebooks: `esda_dependence` (spatial autocorrelation of poverty and lights) and `esda_inequality` (Theil and Gini decompositions). The package's article [Spatial heterogeneity: MGWR](https://quarcs-lab.github.io/geo4bolivia/articles/heterogeneity.html) explains the method step by step, and the [function reference](https://quarcs-lab.github.io/geo4bolivia/reference/mgwr.html) documents every option of `geo.mgwr`.